In [7]:
# CELL 1 - READ RAW PRESCRIPTIONS CSV

from pyspark.sql.functions import current_timestamp, lit

batch_id = "prescriptions_batch_001"
source_file = "prescriptions.csv"

prescriptions_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("Files/raw/prescriptions.csv")
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file_name", lit(source_file))
    .withColumn("ingestion_timestamp", current_timestamp())
)

print("Rows loaded:", prescriptions_df.count())
prescriptions_df.printSchema()
print(prescriptions_df.columns)

display(prescriptions_df.limit(20))

StatementMeta(, e46b3502-6b4d-46a8-a124-d9416e138115, 9, Finished, Available, Finished, False)

Rows loaded: 486634
root
 |-- prescription_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- prescribing_provider_id: string (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- prescriptiontimestamp: string (nullable = true)
 |-- drugndccode: string (nullable = true)
 |-- drug_name: string (nullable = true)
 |-- dosage: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- refills_authorized: string (nullable = true)
 |-- dispense_s: string (nullable = true)
 |-- denial_reason: string (nullable = true)
 |-- pharmacy_ip_address: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['prescription_id', 'patient_id', 'prescribing_provider_id', 'pharmacy_id', 'prescriptiontimestamp', 'drugndccode', 'drug_name', 'dosage', 'quantity', 'refills_authorized', 'dispense_s

SynapseWidget(Synapse.DataFrame, ccc42344-a9b5-479d-84f4-fa100625f8a7)

In [8]:
# CELL 2 — Standardize only safe column formatting

prescriptions_df = prescriptions_df.toDF(
    *[
        column.strip().lower()
        for column in prescriptions_df.columns
    ]
)

prescriptions_df.printSchema()
print(prescriptions_df.columns)

StatementMeta(, e46b3502-6b4d-46a8-a124-d9416e138115, 10, Finished, Available, Finished, False)

root
 |-- prescription_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- prescribing_provider_id: string (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- prescriptiontimestamp: string (nullable = true)
 |-- drugndccode: string (nullable = true)
 |-- drug_name: string (nullable = true)
 |-- dosage: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- refills_authorized: string (nullable = true)
 |-- dispense_s: string (nullable = true)
 |-- denial_reason: string (nullable = true)
 |-- pharmacy_ip_address: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['prescription_id', 'patient_id', 'prescribing_provider_id', 'pharmacy_id', 'prescriptiontimestamp', 'drugndccode', 'drug_name', 'dosage', 'quantity', 'refills_authorized', 'dispense_s', 'denial_reason', 

In [9]:
# CELL 3 — Write prescriptions to Bronze Delta table

spark.sql("DROP TABLE IF EXISTS bronze_prescriptions")

(
    prescriptions_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_prescriptions")
)

print(
    "Bronze rows:",
    spark.table("bronze_prescriptions").count()
)

print("bronze_prescriptions created successfully.")

StatementMeta(, e46b3502-6b4d-46a8-a124-d9416e138115, 11, Finished, Available, Finished, True)

Bronze rows: 486634
bronze_prescriptions created successfully.


In [10]:
# CELL 4 — Verify Bronze prescriptions table

bronze_prescriptions_df = spark.table("bronze_prescriptions")

bronze_prescriptions_df.printSchema()
print(bronze_prescriptions_df.columns)
display(bronze_prescriptions_df.limit(20))

StatementMeta(, e46b3502-6b4d-46a8-a124-d9416e138115, 12, Finished, Available, Finished, True)

root
 |-- prescription_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- prescribing_provider_id: string (nullable = true)
 |-- pharmacy_id: string (nullable = true)
 |-- prescriptiontimestamp: string (nullable = true)
 |-- drugndccode: string (nullable = true)
 |-- drug_name: string (nullable = true)
 |-- dosage: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- refills_authorized: string (nullable = true)
 |-- dispense_s: string (nullable = true)
 |-- denial_reason: string (nullable = true)
 |-- pharmacy_ip_address: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['prescription_id', 'patient_id', 'prescribing_provider_id', 'pharmacy_id', 'prescriptiontimestamp', 'drugndccode', 'drug_name', 'dosage', 'quantity', 'refills_authorized', 'dispense_s', 'denial_reason', 'ph

SynapseWidget(Synapse.DataFrame, b691db31-ccc2-4811-bbc2-bce2ef65ee52)

In [11]:
# CELL 5 — Validate Bronze prescriptions ingestion

source_row_count = prescriptions_df.count()
bronze_row_count = bronze_prescriptions_df.count()

print("Source rows:", source_row_count)
print("Bronze rows:", bronze_row_count)

if source_row_count == bronze_row_count:
    print("Validation passed: all prescription rows reached Bronze.")
else:
    print("Validation failed: row counts do not match.")

StatementMeta(, e46b3502-6b4d-46a8-a124-d9416e138115, 13, Finished, Available, Finished, True)

Source rows: 486634
Bronze rows: 486634
Validation passed: all prescription rows reached Bronze.
